# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jvssankar/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My task is ranking/scoring. I want to prioritize content pages for refresh review so editors can review the most relevant pages first. The score will help order pages using available signals such as traffic, content age, update history, and search performance. The output is decision-support, not an automatic decision to update a page.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
task_type = "ranking/scoring"
decision = "Which content pages should editors review first?"

assert task_type in ["classification", "clustering", "ranking/scoring"]

print("ML task:", task_type)
print("Decision:", decision)

ML task: ranking/scoring
Decision: Which content pages should editors review first?


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target proxy is whether a content page is classified as declining. I will use `trend_direction == "down"` to create a binary label: 1 for down and 0 otherwise. This label is defined by an existing rule using recent and previous 30-day impressions, rather than an independently observed future outcome. Therefore, the results will be interpreted as decision-support for prioritizing pages, not proof of future-decline prediction.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from pathlib import Path

data_path = Path("data/raw/content_refresh_anonymized.csv")

data_url = (
    "https://raw.githubusercontent.com/"
    "Jvssankar/flyrank-ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

if data_path.exists():
    df = pd.read_csv(data_path)
else:
    df = pd.read_csv(data_url)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Dataset shape:", df.shape)
print("\nDecline label counts:")
print(df["is_declining_label"].value_counts().sort_index())

print("\nDecline label percentages:")
print(
    (df["is_declining_label"].value_counts(normalize=True) * 100)
    .sort_index()
    .round(2)
)

Dataset shape: (30000, 45)

Decline label counts:
is_declining_label
0    13738
1    16262
Name: count, dtype: int64

Decline label percentages:
is_declining_label
0    45.79
1    54.21
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My success metric is Precision@50 because the goal is to prioritize pages for editorial review. A provisional good result is at least 0.30, meaning at least 15 of the top 50 pages are labelled declining. This would improve on the repository's reference rule baseline of approximately 0.24. I will confirm the comparison using a held-out client split. This metric measures agreement with the current decline-label rule, not future performance.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
baseline_precision_at_50 = 0.24
success_threshold = 0.30
top_k = 50

required_positive_pages = int(success_threshold * top_k)

print("Metric: Precision@50")
print("Reference baseline:", baseline_precision_at_50)
print("Provisional success threshold:", success_threshold)
print("Positive pages needed in top 50:", required_positive_pages)
print(
    "Required absolute improvement:",
    round(success_threshold - baseline_precision_at_50, 2)
)

Metric: Precision@50
Reference baseline: 0.24
Provisional success threshold: 0.3
Positive pages needed in top 50: 15
Required absolute improvement: 0.06


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one pseudonymized content page. Each row contains page-level information such as content type, age, update history, search impressions, clicks, CTR, and position, along with the decline-label proxy. I will rank candidate pages rather than individual search queries. Pseudonymous IDs are used for identification and grouping, not as predictive features.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
columns_to_show = [
    "content_id",
    "client_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "trend_direction",
    "is_declining_label"
]

lane_df = df[columns_to_show].copy()

print("Rows:", len(lane_df))
print("Unique content pages:", lane_df["content_id"].nunique())
print("Unique clients:", lane_df["client_id"].nunique())

display(lane_df.head(10))

Rows: 30000
Unique content pages: 30000
Unique clients: 32


,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,29,17,0.76,10.6,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,7,9,0.05,20.3,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,11,11,0.09,36.5,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,58,78,0.49,6.2,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,24,145,0.13,44.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,1,5,0.03,8.5,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,0,1,0.00,7.0,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,1,28,0.06,21.2,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,32574,29,68,0.09,46.0,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,1240,2,3,0.16,4.9,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single if-statement may miss combinations of signals such as content age, update history, traffic, CTR, position, and content type. A model may combine these signals to prioritize pages more effectively than a simple rule. However, ML is useful only if it improves Precision@50 on a fair held-out client split compared with the rule baseline. Since my target is derived from an existing decline rule, I will treat the results as decision-support and will not claim proven future-decline prediction.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
group_check = (
    df.groupby(
        ["content_type", "freshness_tier"],
        dropna=False
    )["is_declining_label"]
    .agg(["count", "mean"])
    .reset_index()
)

group_check = group_check.rename(
    columns={
        "count": "pages",
        "mean": "decline_share"
    }
)

group_check["decline_share"] = (
    group_check["decline_share"].round(3)
)

display(
    group_check.sort_values(
        "decline_share",
        ascending=False
    ).head(15)
)

,content_type,freshness_tier,pages,decline_share
1,comparison article,31-90,1,1.000
8,keyword article,91-180,8906,0.612
7,keyword article,31-90,173,0.590
0,comparison article,0-30,696,0.572
4,feedly article,91-180,265,0.570
5,keyword article,0-30,17954,0.536
6,keyword article,181+,174,0.471
2,feedly article,0-30,1830,0.246
3,feedly article,31-90,1,0.000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.